# Notebook 2: GEDI UOI Signal Generation (GEE)

This notebook computes the raw Understory Openness Index (UOI) from GEDI L2B data.
It follows a strict modular structure:
1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee
import geemap

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("\u2713 GEE initialized successfully!")

# Output destination
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
STUDY_REGION = ee.FeatureCollection([
    ee.Feature(CONGO_BBOX, {'basin': 'Congo'}),
    ee.Feature(AMAZON_BBOX, {'basin': 'Amazon'})
])

# Export resolution: 1km captures all GEDI spatial information.
# Multi-scale aggregation (5-100km) is handled downstream in NB3.
EXPORT_SCALE = 1000

# Datasets
GEDI_L2B = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
SRTM = 'USGS/SRTMGL1_003'

# Date range
START_DATE = '2020-01-01'
END_DATE = '2023-12-31'

print("\u2713 Configuration loaded.")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

def load_base_masks():
    """Loads JRC TMF intact forest mask and SRTM topographic filters.
    
    Applies only native-resolution masks (30m JRC, 30m SRTM) to preserve
    maximum data fidelity. The 500m pristine forest mask used for FRIP
    harmonization is applied downstream in NB3 when joining signals.
    """
    # Load 30m JRC TMF
    tmf_col = ee.ImageCollection(FOREST_MASK)
    tmf = tmf_col.mosaic()
    forest_mask = tmf.eq(FOREST_CLASS)
    
    # Topography (Elevation < 1000m, Slope < 10 deg)
    srtm = ee.Image(SRTM)
    elev = srtm.select('elevation')
    slope = ee.Terrain.slope(srtm)
    topo_mask = elev.lt(1000).And(slope.lt(10))
    
    # Combined native mask: intact forest + flat terrain
    combined_mask = forest_mask.And(topo_mask)
    return combined_mask

def compute_native_uoi(combined_mask):
    """Calculates native resolution UOI and observation counts from GEDI L2B.
    
    Returns a two-band image (UOI_mean, N) at GEDI native 25m resolution,
    masked to intact forest on flat terrain.
    """
    gedi = ee.ImageCollection(GEDI_L2B).filterDate(START_DATE, END_DATE)
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        uoi = uoi.clamp(0, 1).rename('UOI')
        return uoi.updateMask(combined_mask)
    
    uoi_col = gedi.map(calc_uoi)
    
    mean_uoi = uoi_col.mean().rename('UOI_mean')
    count_n = uoi_col.count().rename('N')
    
    native_proj = gedi.first().projection()
    native_stack = ee.Image.cat([mean_uoi, count_n]).setDefaultProjection(native_proj)
    
    return native_stack, native_proj

def build_gedi_asset():
    """Builds the GEDI UOI asset at 1km export resolution.
    
    Aggregates 25m GEDI data to 1km using reduceResolution.
    At 25m -> 1km, each output pixel contains ~1,600 input pixels,
    safely under the 65,535 maxPixels limit.
    """
    combined_mask = load_base_masks()
    native_stack, native_proj = compute_native_uoi(combined_mask)
    
    target_proj = native_proj.atScale(EXPORT_SCALE)
    
    # Aggregate 25m -> 1km: mean UOI, sum N
    agg_uoi = native_stack.select('UOI_mean').reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    agg_n = native_stack.select('N').reduceResolution(
        reducer=ee.Reducer.sum(),
        maxPixels=65535
    ).setDefaultProjection(target_proj)
    
    return ee.Image.cat([agg_uoi, agg_n])

print("\u2713 Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running Unit Tests for GEDI UOI generation...")
    passed = 0
    
    try:
        # --- Test 1: Native mask ---
        print("  [1/3] Testing native forest + topo mask construction...")
        combined_mask = load_base_masks()
        assert isinstance(combined_mask, ee.Image), "Mask is not an ee.Image"
        passed += 1
        print("    \u2713 Native mask constructed (30m JRC forest + SRTM topo)")
        
        # --- Test 2: Native UOI computation ---
        print("  [2/3] Testing native UOI computation from GEDI L2B...")
        native_stack, native_proj = compute_native_uoi(combined_mask)
        assert isinstance(native_stack, ee.Image), "Native stack is not an ee.Image"
        
        native_bands = native_stack.bandNames().getInfo()
        assert 'UOI_mean' in native_bands, f"Missing UOI_mean. Got: {native_bands}"
        assert 'N' in native_bands, f"Missing N. Got: {native_bands}"
        passed += 1
        print(f"    \u2713 Native stack bands: {native_bands}")
        
        # --- Test 3: Full asset build at 1km ---
        print("  [3/3] Testing 1km aggregated asset construction...")
        gedi_asset = build_gedi_asset()
        assert isinstance(gedi_asset, ee.Image), "Output is not an ee.Image"
        
        bands = gedi_asset.bandNames().getInfo()
        assert len(bands) == 2, f"Expected 2 bands, got {len(bands)}"
        assert 'UOI_mean' in bands, "Missing UOI_mean band"
        assert 'N' in bands, "Missing N band"
        passed += 1
        print(f"    \u2713 Aggregated asset bands: {bands}")
        
        print(f"\n{'='*60}")
        print(f"  \u2713 ALL {passed} TESTS PASSED")
        print(f"  Server-side execution validated by batch export (Block 4).")
        print(f"  GEE interactive limits prevent deep testing of the 25m\u21921km")
        print(f"  reduceResolution chain, but batch exports have higher budgets.")
        print(f"{'='*60}")
        
    except AssertionError as e:
        print(f"\n  \u2717 Test Failed ({passed} passed): {e}")
    except Exception as e:
        print(f"\n  \u2717 Unexpected Error ({passed} passed): {e}")

# Execute tests
run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXECUTION (ASSET EXPORT)
# =============================================================================

def safe_start(task, asset_id):
    """Deletes existing asset if present, then starts the export task."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"  Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_gedi(dry_run=True):
    print("Configuring GEDI UOI export (single 1km asset)...")
    
    gedi_asset = build_gedi_asset()
    
    asset_id = f'{ASSET_ROOT}/Openness_raw/GEDI_1km'
    task = ee.batch.Export.image.toAsset(
        image=gedi_asset,
        description='GEDI_1km_Export',
        assetId=asset_id,
        region=STUDY_REGION.geometry(),
        scale=EXPORT_SCALE,
        crs='EPSG:4326',
        maxPixels=1e13
    )
    
    print(f"\u2713 Export configured: {asset_id}")
    if dry_run:
        print("DRY RUN: Task created but not started. Call export_gedi(dry_run=False) to begin.")
    else:
        safe_start(task, asset_id)
        print(f"\u2713 Task started! Monitor at https://code.earthengine.google.com/tasks")

# To execute the export, set dry_run=False
export_gedi(dry_run=True)